# MRT analysis: 2024 IHS cohort

* **H1:** a notification improves the proximal outcome in its own domain vs. no notification.
* **H2:** effects on the proximal outcome differ by therapeutic strategy.

Sections: setup, helper functions, data checks and variables, then one section per hypothesis.

## Setup

In [ ]:
library(yaml)
library(here)
library(readr)
library(dplyr)
library(tibble)
library(geepack)

remotes::install_github("WHD-Lab/DR_WCLS_LASSO@2295196c5f364302b0f9b29a154fbc77037daff8", dependencies = FALSE, upgrade = "never")
library(MRTpostInfLASSO)

DATA_PATH <- read_yaml(here("config.yml"))$data_path
OUT_DIR_H1 <- file.path(DATA_PATH, "results", "H1")
OUT_DIR_H2 <- file.path(DATA_PATH, "results", "H2")
for (p in c(OUT_DIR_H1, OUT_DIR_H2)) dir.create(p, recursive = TRUE, showWarnings = FALSE)

SEED   <- 2024
FOLDS  <- 5
METHOD <- "RandomForestLean"  # folds run one at a time, outcome model only; or package methods:
                              # "RandomForest", "CVLASSO", "GradientBoosting" (all folds in parallel)
CORES  <- 4                   # Lean: threads for ranger; package methods: parallel workers (one data copy each)

data <- read_csv(file.path(DATA_PATH, "processed/data_final_2024.csv"), show_col_types = FALSE)

## Helper functions

### Shared

In [ ]:
# Keep one row per decision.
#   device = "pooled": combined Healthkit row for Apple users (watch/phone rows dropped);
#            if a participant has data from several device families, keep the one with
#            the most hours of step data over the study.
#   device = "Fitbit" / "Garmin" / "Healthkit" / ...: only that device's rows.
select_device_rows <- function(data, device = "pooled") {
  d <- data %>% filter(has_decision)
  if (device == "pooled") {
    d <- d %>% filter(!device_type %in% c("HealthkitWatch", "HealthkitPhone"))
    keep <- d %>%
      group_by(participant_id, device_type) %>%
      summarise(cov = sum(steps_n_post, na.rm = TRUE), .groups = "drop") %>%
      group_by(participant_id) %>%
      slice_max(cov, n = 1, with_ties = FALSE) %>%
      ungroup() %>%
      select(participant_id, device_type)
    d <- d %>% semi_join(keep, by = c("participant_id", "device_type"))
  } else {
    d <- d %>% filter(device_type == device)
  }
  stopifnot(!anyDuplicated(d$decision_id))
  d
}

In [ ]:
# Coverage and window-length rules (analysis choices; vary them in sensitivity analyses).
#   Steps and sleep totals (and log steps) are set to NA when the window has data for
#   fewer than `min_cov` of its hours (`<stream>_cov_<w>` from preprocessing) or its
#   length is outside `span_range`; mood is set to NA only for the window length rule.
apply_thresholds <- function(d, min_cov = 0.75, span_range = c(20, 28)) {
  for (w in c("pre", "post")) {
    span <- d[[paste0("span_h_", w)]]
    ok_span <- !is.na(span) & span >= span_range[1] & span <= span_range[2]
    for (s in c("steps", "sleep_minutes")) {
      cov <- d[[paste0(s, "_cov_", w)]]
      ok <- ok_span & !is.na(cov) & cov >= min_cov
      d[[paste0(s, "_", w)]][!ok] <- NA
      if (s == "steps") d[[paste0("log_steps_", w)]][!ok] <- NA
    }
    d[[paste0("Mood_", w)]][!ok_span] <- NA
  }
  d
}

In [ ]:
# Median-impute history variables and add missingness indicators (the nuisance
# learners cannot take NA). Uses only history, never the outcome.
impute_history <- function(d, Ht) {
  out <- Ht
  for (v in Ht) {
    miss <- is.na(d[[v]])
    if (any(miss)) {
      d[[paste0(v, "_miss")]] <- as.integer(miss)
      d[[v]][miss] <- median(d[[v]], na.rm = TRUE)
      out <- c(out, paste0(v, "_miss"))
    }
  }
  list(data = d, Ht = out)
}

### H1

In [ ]:
# Treatment, randomization probability and analysis rows for one contrast.
#   domain = "any": any message vs no message, P(A = 1) = p_send.
#   domain = "mood"/"steps"/"sleep": a message of that domain vs no message; decisions
#   with a message of another domain are dropped, so
#   P(A = 1 | H_t, kept) = p_send * p_domain / (p_send * p_domain + p_no_send).
code_treatment <- function(d, domain, metric_values) {
  if (domain == "any") {
    d$A <- d$action_id
    d$prob <- d$p_send
  } else {
    m <- metric_values[[domain]]
    d <- d %>% filter(action_id == 0 | message_metric == m)
    p_dom <- d[[paste0("p_", domain)]]
    d$A <- d$action_id
    d$prob <- d$p_send * p_dom / (d$p_send * p_dom + d$p_no_send)
  }
  stopifnot(all(d$A %in% c(0, 1)), all(d$prob > 0 & d$prob < 1))
  d
}

In [ ]:
# Cross-fitted nuisance models from MRTpostInfLASSO, then the DR pseudo-outcome with
# the known randomization probability and a constant reference probability
# p_tilde = mean(prob) (S_t is empty for the marginal effect).
PRED_COLS <- list(
  RandomForest     = c("gt_pred_rf", "gtAt1_pred_rf", "gtAt0_pred_rf"),
  GradientBoosting = c("gt_pred_gbm", "gtAt1_pred_gbm", "gtAt0_pred_gbm"),
  CVLASSO          = c("lasso.pred.gt", "lasso.pred.gtAt1", "lasso.pred.gtAt0")
)

In [ ]:
# Lean cross-fitting with ranger: same outcome model as the package
# (Y ~ Ht + A + Ht:A, participants split into folds), but folds run one at a time,
# only the outcome model is fit (treatment probabilities are known), and forest
# size is controllable, so memory stays bounded.
crossfit_rf_lean <- function(d, Ht, outcome, fold, num_threads, seed,
                             num_trees = 300, min_node_size = 50) {
  # split participants into folds (same scheme as the package's internal split_data)
  set.seed(seed)
  ids <- sample(unique(d$participant_id))
  folds <- split(ids, rep(seq_len(fold), length.out = length(ids)))
  d <- as.data.frame(d)
  add_inter <- function(x) {
    for (v in Ht) x[[paste0(v, ".A")]] <- x[[v]] * x$A
    x
  }
  feats <- c(Ht, "A", paste0(Ht, ".A"))
  f <- as.formula(paste(outcome, "~", paste(feats, collapse = " + ")))
  out <- vector("list", length(folds))
  for (k in seq_along(folds)) {
    test <- d$participant_id %in% folds[[k]]
    rf <- ranger::ranger(f, data = add_inter(d[!test, c(outcome, Ht, "A")]),
                         num.trees = num_trees, min.node.size = min_node_size,
                         num.threads = num_threads, seed = seed + k,
                         write.forest = TRUE, verbose = FALSE)
    te <- d[test, ]
    te1 <- te; te1$A <- 1
    te0 <- te; te0$A <- 0
    te$gt_pred_rf    <- predict(rf, add_inter(te[,  c(Ht, "A")]), num.threads = num_threads)$predictions
    te$gtAt1_pred_rf <- predict(rf, add_inter(te1[, c(Ht, "A")]), num.threads = num_threads)$predictions
    te$gtAt0_pred_rf <- predict(rf, add_inter(te0[, c(Ht, "A")]), num.threads = num_threads)$predictions
    out[[k]] <- te
    rm(rf); gc(verbose = FALSE)
  }
  do.call(rbind, out)
}

dr_pseudo_outcomes <- function(d, Ht, outcome, method, fold, core_num, seed) {
  if (method == "RandomForestLean") {
    ps <- crossfit_rf_lean(d, Ht, outcome, fold, num_threads = core_num, seed = seed)
    return(finish_pseudo(ps, d, outcome, PRED_COLS[["RandomForest"]]))
  }
  gen <- switch(method,
    RandomForest     = pseudo_outcome_generator_rf_v2,
    GradientBoosting = pseudo_outcome_generator_gbm,
    CVLASSO          = pseudo_outcome_generator_CVlasso)
  # The package also fits E[A | S_t] and needs a non-empty S_t (two columns for
  # CVLASSO). That estimate is not used: it is replaced below by the constant p_tilde.
  ps <- gen(fold = fold, ID = "participant_id", data = as.data.frame(d),
            Ht = Ht, St = Ht[1:2], At = "A", prob = "prob", outcome = outcome,
            core_num = core_num, master_seed = seed)
  finish_pseudo(ps, d, outcome, PRED_COLS[[method]])
}

finish_pseudo <- function(ps, d, outcome, pc) {
  A <- ps$A; p <- ps$prob
  p_tilde <- mean(d$prob)
  W <- ifelse(A == 1, p_tilde / p, (1 - p_tilde) / (1 - p))
  ps$p_tilde <- p_tilde
  ps$W <- W
  ps$yDR <- W * (A - p_tilde) * (ps[[outcome]] - ps[[pc[1]]]) / (p_tilde * (1 - p_tilde)) +
    (ps[[pc[2]]] - ps[[pc[3]]])
  ps %>% arrange(participant_id, t)
}

In [ ]:
# Marginal effect: weighted regression of yDR on an intercept, sandwich SE clustered
# by participant (as in the package's GEE step).
fit_marginal <- function(ps, level = 0.95) {
  ps <- ps %>% arrange(participant_id, t)
  wt <- ps$p_tilde * (1 - ps$p_tilde)
  fit <- geepack::geeglm(yDR ~ 1, data = ps, weights = wt,
                         id = factor(participant_id, levels = unique(participant_id)),
                         corstr = "independence")
  tidy_gee(fit, "(Intercept)", level)
}

In [ ]:
# WCLS check: linear control model, centered treatment, weights W.
fit_wcls <- function(d, Ht_lin, outcome, level = 0.95) {
  d <- d %>% arrange(participant_id, t)
  p_tilde <- mean(d$prob)
  d$A_c <- d$A - p_tilde
  d$W <- ifelse(d$A == 1, p_tilde / d$prob, (1 - p_tilde) / (1 - d$prob))
  f <- as.formula(paste(outcome, "~", paste(c(Ht_lin, "A_c"), collapse = " + ")))
  fit <- geepack::geeglm(f, data = d, weights = W,
                         id = factor(participant_id, levels = unique(participant_id)),
                         corstr = "independence")
  tidy_gee(fit, "A_c", level)
}

tidy_gee <- function(fit, term, level) {
  s <- summary(fit)$coefficients
  est <- s[term, "Estimate"]; se <- s[term, "Std.err"]
  z <- qnorm(1 - (1 - level) / 2)
  tibble(estimate = est, se = se, lower = est - z * se, upper = est + z * se,
         p_value = 2 * pnorm(-abs(est / se)))
}

In [ ]:
# One H1 analysis: contrast for `domain`, outcome `outcome`.
run_h1 <- function(data, domain, outcome, Ht, Ht_lin, metric_values,
                   device = "pooled", method = "RandomForest",
                   fold = 5, core_num = 4, seed = 2024, min_participants = 20) {
  d <- data %>%
    select_device_rows(device) %>%
    code_treatment(domain, metric_values)
  n_all <- nrow(d)
  d <- d %>% filter(!is.na(.data[[outcome]]))
  n_id <- n_distinct(d$participant_id)
  if (n_id < max(fold, min_participants)) {
    message(sprintf("skipping %s / %s / %s: %d participants with outcome data (need %d)",
                    domain, outcome, device, n_id, max(fold, min_participants)))
    return(list(results = tibble(domain = domain, outcome = outcome, device = device,
                                 n_participants = n_id, n_decisions = nrow(d),
                                 estimator = NA_character_, estimate = NA_real_, se = NA_real_,
                                 lower = NA_real_, upper = NA_real_, p_value = NA_real_),
                pseudo = NULL))
  }
  imp <- impute_history(d, Ht)
  d <- imp$data

  ps <- dr_pseudo_outcomes(d, imp$Ht, outcome, method, fold, core_num, seed)
  sd_y <- sd(d[[outcome]][d$A == 0])

  res <- bind_rows(
    fit_marginal(ps) %>% mutate(estimator = paste0("DR-WCLS (", method, ")")),
    fit_wcls(d, Ht_lin, outcome) %>% mutate(estimator = "WCLS (linear control)")
  ) %>%
    mutate(domain = domain, outcome = outcome, device = device,
           n_participants = n_distinct(d$participant_id),
           n_decisions = nrow(d), n_missing_outcome = n_all - nrow(d),
           p_treated = mean(d$A), p_tilde = mean(d$prob),
           std_effect = estimate / sd_y,
           .before = 1)
  list(results = res, pseudo = ps)
}

### H2

In [ ]:
# Arms for one H2 analysis: "none" (no message) plus one arm per strategy.
#   mode = "within_domain": only no-message decisions and messages of `domain` are kept;
#     P(strategy k | H_t, kept) = p_send * p_domain * p_k / (p_no_send + p_send * p_domain).
#     Assumes p_k (from `strategy_probs`) is P(strategy k | send) and that strategy and
#     domain are drawn independently -- check this before use.
#   mode = "any_domain": all decisions; P(strategy k | H_t) = p_send * p_k.
# Arms with fewer than `min_arm_n` decisions are dropped, and so are decisions where any
# remaining arm has probability below `min_prob` (positivity).
code_arms <- function(d, domain, mode, metric_values, strategy_probs,
                      min_arm_n = 200, min_prob = 1e-3) {
  if (mode == "within_domain") {
    d <- d %>% filter(action_id == 0 | message_metric == metric_values[[domain]])
    p_dom <- d[[paste0("p_", domain)]]
  } else {
    p_dom <- 1
  }
  keys <- strategy_key(names(strategy_probs))
  d$arm <- ifelse(d$action_id == 0, "none", strategy_key(d$strategy))

  P <- sapply(names(strategy_probs), function(s) d$p_send * p_dom * d[[strategy_probs[[s]]]])
  P <- cbind(d$p_no_send, matrix(P, nrow = nrow(d)))
  colnames(P) <- c("none", keys)

  counts <- table(factor(d$arm, levels = colnames(P)))
  small <- setdiff(names(counts)[counts < min_arm_n], "none")
  if (length(small)) {
    message("dropping arms with < ", min_arm_n, " decisions: ", paste(small, collapse = ", "))
    keep_row <- !d$arm %in% small
    d <- d[keep_row, ]; P <- P[keep_row, setdiff(colnames(P), small), drop = FALSE]
  }
  P <- P / rowSums(P)
  ok <- apply(P, 1, min) >= min_prob
  if (any(!ok)) {
    message(sum(!ok), " decisions dropped: an arm had probability < ", min_prob)
    d <- d[ok, ]; P <- P[ok, , drop = FALSE]
  }
  for (a in colnames(P)) d[[paste0("pi_", a)]] <- P[, a]
  stopifnot(all(d$arm %in% colnames(P)))
  attr(d, "arms") <- colnames(P)
  d
}

strategy_key <- function(x) gsub("^_|_$", "", gsub("[^a-z0-9]+", "_", tolower(x)))

# Cross-fitted outcome models, one random forest per arm (fit on that arm's training
# rows), participants split into folds; adds g_<arm> = predicted outcome under each arm.
crossfit_multiarm <- function(d, Ht, outcome, arms, fold, num_threads, seed,
                              num_trees = 300, min_node_size = 50) {
  set.seed(seed)
  ids <- sample(unique(d$participant_id))
  folds <- split(ids, rep(seq_len(fold), length.out = length(ids)))
  d <- as.data.frame(d)
  f <- as.formula(paste(outcome, "~", paste(Ht, collapse = " + ")))
  out <- vector("list", length(folds))
  for (k in seq_along(folds)) {
    test <- d$participant_id %in% folds[[k]]
    te <- d[test, ]
    for (a in arms) {
      tr <- d[!test & d$arm == a, c(outcome, Ht)]
      rf <- ranger::ranger(f, data = tr, num.trees = num_trees, min.node.size = min_node_size,
                           num.threads = num_threads, seed = seed + k, verbose = FALSE)
      te[[paste0("g_", a)]] <- predict(rf, te[, Ht], num.threads = num_threads)$predictions
      rm(rf); gc(verbose = FALSE)
    }
    out[[k]] <- te
  }
  do.call(rbind, out) %>% arrange(participant_id, t)
}

In [ ]:
# Doubly robust pseudo-outcome for each strategy k vs no message:
#   yDR_k = 1{A = k}/pi_k (Y - g_k) - 1{A = none}/pi_none (Y - g_none) + g_k - g_none
multiarm_pseudo <- function(ps, outcome, arms) {
  y <- ps[[outcome]]
  r0 <- (ps$arm == "none") / ps$pi_none * (y - ps$g_none)
  for (a in setdiff(arms, "none")) {
    ps[[paste0("yDR_", a)]] <- (ps$arm == a) / ps[[paste0("pi_", a)]] * (y - ps[[paste0("g_", a)]]) -
      r0 + ps[[paste0("g_", a)]] - ps$g_none
  }
  ps
}

In [ ]:
# Effects of each strategy vs no message (means of yDR_k) with a sandwich covariance
# clustered by participant across all arms and decisions, a Wald test that all
# strategy effects are equal, and pairwise differences (Holm-adjusted p-values).
fit_multiarm <- function(ps, arms, level = 0.95) {
  strat <- setdiff(arms, "none")
  long <- bind_rows(lapply(strat, function(a)
    tibble(participant_id = ps$participant_id, t = ps$t, arm = a, yDR = ps[[paste0("yDR_", a)]]))) %>%
    mutate(arm = factor(arm, levels = strat)) %>%
    arrange(participant_id, t, arm)
  fit <- geepack::geeglm(yDR ~ 0 + arm, data = long,
                         id = factor(participant_id, levels = unique(participant_id)),
                         corstr = "independence")
  b <- coef(fit); V <- fit$geese$vbeta
  names(b) <- strat; dimnames(V) <- list(strat, strat)
  z <- qnorm(1 - (1 - level) / 2)
  se <- sqrt(diag(V))
  effects <- tibble(strategy = strat, estimate = b, se = se,
                    lower = b - z * se, upper = b + z * se, p_value = 2 * pnorm(-abs(b / se)))

  K <- length(strat)
  test <- tibble(test = "all strategy effects equal", df = K - 1, chisq = NA_real_, p_value = NA_real_)
  pairs <- tibble()
  if (K >= 2) {
    C <- cbind(-1, diag(K - 1))
    cb <- C %*% b
    chisq <- drop(t(cb) %*% solve(C %*% V %*% t(C)) %*% cb)
    test$chisq <- chisq; test$p_value <- pchisq(chisq, K - 1, lower.tail = FALSE)
    cmb <- combn(K, 2)
    pairs <- bind_rows(lapply(seq_len(ncol(cmb)), function(j) {
      i1 <- cmb[1, j]; i2 <- cmb[2, j]
      est <- b[i1] - b[i2]; s <- sqrt(V[i1, i1] + V[i2, i2] - 2 * V[i1, i2])
      tibble(contrast = paste(strat[i1], "-", strat[i2]), estimate = est, se = s,
             lower = est - z * s, upper = est + z * s, p_value = 2 * pnorm(-abs(est / s)))
    })) %>% mutate(p_holm = p.adjust(p_value, "holm"))
  }
  list(effects = effects, test = test, pairwise = pairs)
}

In [ ]:
# One H2 analysis.
run_h2 <- function(data, domain, outcome, Ht, metric_values, strategy_probs,
                   mode = "within_domain", device = "pooled",
                   fold = 5, core_num = 4, seed = 2024, min_participants = 20) {
  d <- data %>%
    select_device_rows(device) %>%
    code_arms(domain, mode, metric_values, strategy_probs)
  arms <- attr(d, "arms")
  n_all <- nrow(d)
  d <- d %>% filter(!is.na(.data[[outcome]]))
  if (n_distinct(d$participant_id) < max(fold, min_participants) || length(arms) < 2) {
    message(sprintf("skipping H2 %s / %s / %s: too few participants or arms", domain, outcome, device))
    return(NULL)
  }
  imp <- impute_history(d, Ht)
  ps <- crossfit_multiarm(imp$data, imp$Ht, outcome, arms, fold, core_num, seed) %>%
    multiarm_pseudo(outcome, arms)
  fit <- fit_multiarm(ps, arms)

  info <- tibble(domain = domain, outcome = outcome, mode = mode, device = device)
  n_arm <- d %>% count(arm, name = "n_decisions")
  sd_y <- sd(d[[outcome]][d$arm == "none"])
  list(
    effects  = bind_cols(info, fit$effects) %>%
      left_join(n_arm, by = c("strategy" = "arm")) %>%
      mutate(std_effect = estimate / sd_y),
    test     = bind_cols(info, fit$test) %>%
      mutate(n_participants = n_distinct(d$participant_id), n_decisions = nrow(d),
             n_missing_outcome = n_all - nrow(d)),
    pairwise = if (nrow(fit$pairwise)) bind_cols(info, fit$pairwise) else fit$pairwise,
    pseudo   = ps
  )
}

## Data checks and variables

### Treatment and probability columns

In [ ]:
print(table(data$message_metric, useNA = "ifany"))
print(table(data$device_type, useNA = "ifany"))

# map domains to the values used in `message_metric`
METRIC_VALUES <- list(mood = "Mood", steps = "Step Count", sleep = "Hours of Sleep")
stopifnot(all(unlist(METRIC_VALUES) %in% data$message_metric))
stopifnot(all(c("p_send", "p_no_send", paste0("p_", names(METRIC_VALUES))) %in% names(data)))

# do the domain probabilities vary within participants? (messages drawn without replacement)
data %>%
  filter(has_decision) %>%
  group_by(participant_id) %>%
  summarise(across(c(p_mood, p_steps, p_sleep), sd), .groups = "drop") %>%
  summarise(across(-participant_id, ~ mean(.x > 1e-8, na.rm = TRUE)))

### Outcomes and history

Log steps, coverage shares (`<stream>_cov_<window>`) and local day of week come from preprocessing.
Here, step and sleep totals are set to missing when the window has data for fewer than
75% of its hours or is outside 20–28 hours long; mood is set to missing for the length rule only.
History variables are median-imputed with missingness indicators inside `run_h1`.

In [ ]:
data <- apply_thresholds(data, min_cov = 0.75, span_range = c(20, 28))

OUTCOMES <- c(mood = "Mood_post", steps = "log_steps_post", sleep = "sleep_minutes_post")

# TODO: decide whether to include day of week (local time) or a weekend indicator
Ht_candidates <- c(
  "Mood_pre", "Mood_sd_pre", "log_steps_pre", "sleep_minutes_pre", "hr_avg_pre", "RHR_pre",
  "Mood_n_pre", "steps_n_pre", "sleep_minutes_n_pre",
  "t", #"dow", "weekend",
  "Age", "Sex", "depr0", "PHQtot0", "GADtot0", "Neu0_published", "SLE0"
)
missing_cols <- setdiff(Ht_candidates, names(data))
if (length(missing_cols)) message("not in data, skipped: ", paste(missing_cols, collapse = ", "))
Ht <- intersect(Ht_candidates, names(data))

# small linear control model for the WCLS check
Ht_lin <- intersect(c("Mood_pre", "log_steps_pre", "sleep_minutes_pre", "t", "PHQtot0"), Ht)

# summary of missing outcomes by domain
sapply(OUTCOMES, function(y) mean(is.na(data[[y]][data$has_decision])))

## H1: effect of a notification on the proximal outcome in its own domain

For each domain d (mood, steps, sleep), the causal excursion effect of sending a
domain-d message vs. no message on the domain-d outcome in the following window,
averaged over all decision points (S_t empty):

* **Rows:** decisions with no message or a domain-d message (other-domain messages dropped).
* **Treatment probability:** P(A = 1 | H_t) = p_send · p_d / (p_send · p_d + p_no_send), known from the design.
* **Estimator:** DR-WCLS. Nuisance outcome models are cross-fitted by participant with
  `MRTpostInfLASSO`; the pseudo-outcome uses the known randomization probability and a
  constant reference probability p̃ = mean(P(A = 1)). The marginal effect is a weighted
  regression of the pseudo-outcome on an intercept with sandwich SEs clustered by participant.
* **Check:** WCLS with a linear control model on the same rows.
* **Secondary:** any message vs. no message on mood.

No moderator selection is involved, so the LASSO and post-selection steps of the package are not used here.
Helper functions are under **Helper functions → H1**.

### Primary analysis: pooled devices

In [ ]:
specs <- tibble(
  domain  = c("mood", "steps", "sleep", "any"),
  outcome = c(OUTCOMES[["mood"]], OUTCOMES[["steps"]], OUTCOMES[["sleep"]], OUTCOMES[["mood"]]),
  role    = c("primary", "primary", "primary", "secondary")
)

results <- list()
for (i in seq_len(nrow(specs))) {
  s <- specs[i, ]
  message("H1: ", s$domain, " -> ", s$outcome)
  r <- run_h1(data, s$domain, s$outcome, Ht, Ht_lin, METRIC_VALUES,
              device = "pooled", method = METHOD,
              fold = FOLDS, core_num = CORES, seed = SEED)
  results[[i]] <- r$results %>% mutate(role = s$role, .before = 1)
  saveRDS(r$pseudo, file.path(OUT_DIR_H1, sprintf("pseudo_%s_%s_pooled.rds", s$domain, s$outcome)))
}
h1 <- bind_rows(results)
write_csv(h1, file.path(OUT_DIR_H1, "h1_pooled.csv"))

h1 %>% select(role, domain, outcome, estimator, n_participants, n_decisions,
              estimate, se, lower, upper, p_value, std_effect)

### Sensitivity: device-specific analyses

Device is not randomized, so differences between devices mix measurement with who uses which device.

In [ ]:
RUN_DEVICES <- TRUE
DEVICES <- c("Fitbit", "Garmin", "Healthkit", "HealthkitPhone", "HealthkitWatch")

if (RUN_DEVICES) {
  dev_results <- list()
  for (dv in DEVICES) for (i in which(specs$role == "primary")) {
    s <- specs[i, ]
    r <- run_h1(data, s$domain, s$outcome, Ht, Ht_lin, METRIC_VALUES,
                device = dv, method = METHOD,
                fold = FOLDS, core_num = CORES, seed = SEED)
    dev_results[[length(dev_results) + 1]] <- r$results
  }
  h1_dev <- bind_rows(dev_results)
  write_csv(h1_dev, file.path(OUT_DIR_H1, "h1_by_device.csv"))
  h1_dev %>% select(device, domain, estimator, n_participants, estimate, se, lower, upper, p_value)
}

## H2: effects differ by therapeutic strategy

For each outcome domain d, the effect of a message using strategy k vs. no message, for each strategy, and a test of whether these effects differ:

* **Arms:** no message plus one arm per strategy. In the primary design (`within_domain`), only decisions with no message or a domain-d message are kept, so each arm is "a domain-d message using strategy k".
* **Probabilities:** P(arm k | H_t) = p_send · p_d · p_k / (p_no_send + p_send · p_d), and P(no message | H_t) = p_no_send / (same). This assumes p_k is P(strategy k | send) and that strategy and domain are drawn independently; the check below compares observed domain × strategy counts with what that assumption predicts. If it fails, use `H2_MODE <- "any_domain"`: strategy k messages of any domain vs. no message, with P = p_send · p_k, which does not need the assumption.
* **Estimator:** doubly robust (AIPW) pseudo-outcome for each strategy vs. no message, with cross-fitted outcome models (one random forest per arm). With an empty S_t and a constant reference probability, this is the multi-arm version of the H1 DR-WCLS pseudo-outcome.
* **Inference:** strategy effects are the means of their pseudo-outcomes, with a sandwich covariance clustered by participant across arms and decisions. H2 is tested with a Wald test that all strategy effects are equal; pairwise differences have Holm-adjusted p-values.

### Strategy labels and probability checks

In [ ]:
print(table(data$strategy, useNA = "ifany"))

# map strategy labels to their probability columns; edit if the names differ
strategies <- sort(unique(na.omit(data$strategy)))
STRATEGY_PROBS <- setNames(as.list(paste0("p_", strategy_key(strategies))), strategies)
print(tibble(strategy = strategies, p_col = unlist(STRATEGY_PROBS),
             found = unlist(STRATEGY_PROBS) %in% names(data)))
stopifnot(all(unlist(STRATEGY_PROBS) %in% names(data)))

# strategy probabilities sum to 1 among sends?
sends <- data %>% filter(has_decision, action_id == 1)
summary(rowSums(as.matrix(sends[, unlist(STRATEGY_PROBS)])))

# independence check: observed domain x strategy counts vs. expected sum of p_d * p_k
domain_key <- setNames(names(METRIC_VALUES), unlist(METRIC_VALUES))
obs <- sends %>% count(domain = domain_key[message_metric], strategy, name = "observed")
expd <- bind_rows(lapply(names(METRIC_VALUES), function(dm) bind_rows(lapply(strategies, function(s)
  tibble(domain = dm, strategy = s,
         expected = sum(sends[[paste0("p_", dm)]] * sends[[STRATEGY_PROBS[[s]]]])))))) 
expd %>% left_join(obs, by = c("domain", "strategy")) %>%
  mutate(observed = coalesce(observed, 0L), ratio = observed / expected) %>%
  arrange(domain, strategy)

Ratios near 1 in every cell support the independence assumption. Cells with ratios far from 1
(especially strategies never used in a domain) mean `within_domain` probabilities are wrong;
switch to `H2_MODE <- "any_domain"`.

### Primary analysis: pooled devices

In [ ]:
H2_MODE <- "within_domain"   # or "any_domain" if the independence check fails

h2_specs <- tibble(domain = names(OUTCOMES), outcome = unname(OUTCOMES))
if (H2_MODE == "any_domain") h2_specs <- tidyr::expand_grid(domain = "any", outcome = unname(OUTCOMES))

h2 <- list()
for (i in seq_len(nrow(h2_specs))) {
  s <- h2_specs[i, ]
  message("H2: ", s$domain, " -> ", s$outcome, " (", H2_MODE, ")")
  h2[[i]] <- run_h2(data, s$domain, s$outcome, Ht, METRIC_VALUES, STRATEGY_PROBS,
                    mode = H2_MODE, device = "pooled",
                    fold = FOLDS, core_num = CORES, seed = SEED)
  if (!is.null(h2[[i]]))
    saveRDS(h2[[i]]$pseudo, file.path(OUT_DIR_H2, sprintf("pseudo_%s_%s_%s_pooled.rds", H2_MODE, s$domain, s$outcome)))
}
h2 <- h2[!sapply(h2, is.null)]
h2_effects  <- bind_rows(lapply(h2, `[[`, "effects"))
h2_tests    <- bind_rows(lapply(h2, `[[`, "test"))
h2_pairwise <- bind_rows(lapply(h2, `[[`, "pairwise"))
write_csv(h2_effects,  file.path(OUT_DIR_H2, "h2_effects_pooled.csv"))
write_csv(h2_tests,    file.path(OUT_DIR_H2, "h2_tests_pooled.csv"))
write_csv(h2_pairwise, file.path(OUT_DIR_H2, "h2_pairwise_pooled.csv"))

h2_tests %>% select(domain, outcome, n_participants, n_decisions, df, chisq, p_value)

In [ ]:
h2_effects %>% select(domain, strategy, n_decisions, estimate, se, lower, upper, p_value, std_effect)

In [ ]:
h2_pairwise %>% select(domain, contrast, estimate, se, lower, upper, p_value, p_holm)